# Detecting Illicit Bitcoin Transactions in a Temporal Graph

## Geometric Learning, Time-Variant Data Analysis, and Anomaly Detection

**Dataset:** Elliptic Bitcoin transaction graph  
**Task:** detect illicit transactions under temporal non-stationarity  
**Main thesis:** the catastrophic post-shock failure is mostly a *prior-shift / topology-adaptation problem*, not a collapse of the node representation.

Generated from:

- `results/` experiment outputs
- `source/` implementation
- `source/reporting/results/` analysis summaries


## Executive summary

1. The Elliptic graph is a sequence of directed transaction snapshots $G_\tau=(V_\tau,E_\tau)$.
2. The shock at $\tau=43$ is not primarily a geometric collapse. It is a **class-prior collapse**: illicit labels drop from 239 at $\tau=42$ to 24 at $\tau=43$.
3. SGC propagation and PCA uncover useful graph structure, but graph models overfit to pre-shock micro-motifs.
4. XGBoost is the strongest overall benchmark because node-level tabular structure survives the regime change better than graph motifs.
5. The final MLP-head experiment improved the old graph-MLP validation PR-AUC, but the residual connection itself did not help. The useful head is **LayerNorm + SiLU + small non-residual MLP**.


## Data model and notation

At each time step $\tau \in \{1,\dots,49\}$ we have a directed graph

$$
G_\tau=(V_\tau,E_\tau,X_\tau,y_\tau),
$$

where each node is a Bitcoin transaction, each edge is a flow of funds, and

$$
y_i \in \{0,1,-1\}
$$

means licit, illicit, or unknown. Unknown labels are excluded from the loss and metrics.

The supervised task is imbalanced: the positive class is illicit, so **PR-AUC** is more informative than accuracy.


## The temporal regime split

| $\tau$ | nodes | illicit | illicit rate | mean degree | regime |
| --- | --- | --- | --- | --- | --- |
| 42 | 7,140 | 239 | 0.111 | 2.379 | pre_shock |
| 43 | 5,063 | 24 | 0.018 | 2.350 | shock |
| 44 | 4,975 | 24 | 0.015 | 2.232 | recovery |
| 45 | 5,598 | 5 | 0.004 | 2.384 | recovery |
| 46 | 3,519 | 2 | 0.003 | 2.197 | recovery |
| 49 | 2,454 | 56 | 0.118 | 2.108 | recovery |

The key observation is the discontinuity in illicit prevalence at $\tau=43$. The global graph is still present; the minority class almost disappears.


![Prior shift](assets/01_prior_shift.png)


![Graph stability](assets/02_graph_stability.png)


## Visual diagnostics from the EDA pipeline

These figures were generated earlier in the project and copied from `results/figures/`.

![Ground truth timeline](assets/panel1_ground_truth.png)

![Class imbalance panel](assets/eda_panel_a_imbalance.png)

![Volume panel](assets/eda_panel_b_volume.png)

![Graph hairball panel](assets/eda_panel_c_hairball.png)


## Exploratory graph geometry

The raw feature geometry and local graph structure are informative:

- illicit nodes are more structurally constrained downstream;
- illicit interactions often pass through unknown intermediaries;
- licit transactions include high-degree service-like hubs;
- graph homophily is weak for illicit nodes, which makes naive neighbor smoothing risky.


![Embeddings](assets/04_embeddings.png)


![Homophily and degree](assets/05_homophily_degree.png)


## SGC propagation: the graph signal used by the model

The implementation in `source/models/layers.py` uses a symmetrically normalized adjacency

$$
\tilde A = D^{-1/2}\,(\max(A,A^T)+I)\,D^{-1/2}.
$$

For multiscale SGC, the representation is

$$
\Phi_K(X)=\left[X\;\middle|\;\tilde A X\;\middle|\;\tilde A^2X\;\middle|\;\cdots\;\middle|\;\tilde A^KX\right].
$$

The directional version augments the symmetric channel with outgoing and incoming row-normalized channels:

$$
\Phi_K^{dir}(X)=
\left[
X,
\tilde A_{sym}X,\tilde A_{out}X,\tilde A_{in}X,
\dots,
\tilde A_{sym}^KX,\tilde A_{out}^KX,\tilde A_{in}^KX
\right].
$$

This is not a trainable GCN layer; propagation is deterministic, and the learning happens in the classifier head.


## PCA and oversmoothing

Deep propagation increases the amount of neighborhood information mixed into each node. For large $K$, this can cause **oversmoothing**: node representations become too similar.

The project tested intrinsic dimensionality as a diagnostic. A drop in intrinsic dimension at larger $K$ is evidence that representations are collapsing toward a lower-dimensional, less discriminative manifold.


![Intrinsic dimension](assets/06_intrinsic_dimension.png)


## Falsifying representational collapse at $\tau=43$

The strongest misconception is: "the graph embedding collapses at the shock." The diagnostic files contradict that.

**Feature drift around the shock:**

| $\tau$ | MMD | Wasserstein PCA |
| --- | --- | --- |
| 42 | 0.0034 | 0.9319 |
| 43 | 0.0128 | 1.0706 |
| 44 | 0.0406 | 1.4061 |
| 45 | 0.0150 | 2.5095 |
| 46 | 0.0295 | 2.7901 |

**Permutation separability tests:**

| $\tau$ | representation | n illicit | separable seed fraction | median perm. p |
| --- | --- | --- | --- | --- |
| 42 | Raw | 239.0000 | 1.0000 | 0.0010 |
| 42 | $\tilde A X$ | 239.0000 | 1.0000 | 0.0010 |
| 43 | Raw | 24.0000 | 0.8000 | 0.0120 |
| 43 | $\tilde A X$ | 24.0000 | 1.0000 | 0.0095 |
| 44 | Raw | 24.0000 | 0.5000 | 0.0584 |
| 44 | $\tilde A X$ | 24.0000 | 0.5000 | 0.0689 |
| 45 | Raw | 5.0000 | 0.1000 | 0.4745 |
| 45 | $\tilde A X$ | 5.0000 | 0.0000 | 0.4620 |

The propagated representation at $\tau=43$ remains separable. The model failure is therefore more consistent with **label deprivation and threshold/head-level imbalance** than with geometric collapse.


![Drift and separability](assets/03_drift_separability.png)


## Static evaluation: validation and out-of-time tests

The static protocol trains on early timesteps, validates on an intermediate block, and reports out-of-time performance on future timesteps.

This is where graph propagation helps compared with a plain SGC baseline, but tabular tree models remain very strong.


![Static results](assets/07_static_results.png)


### Static result table

| model | Val Macro PR-AUC | OOT pooled F1 | OOT pooled PR-AUC | OOT macro F1 |
| --- | --- | --- | --- | --- |
| RandomForest | 0.985 | 0.777 | 0.783 | 0.479 |
| XGBoost | 0.976 | 0.765 | 0.790 | 0.475 |
| PyG GCN | 0.799 | 0.480 | 0.448 | 0.287 |
| Old SGC+MLP | 0.815 | 0.455 | 0.424 | 0.261 |
| Best old Graph Grid | 0.857 | 0.477 | 0.442 | 0.270 |
| Final LN+SiLU MLP | 0.881 | 0.471 | 0.452 | 0.258 |


## Walk-forward validation

The implementation in `source/evaluation/validation.py` uses a leakage-guarded walk-forward protocol:

$$
\text{train on } [1,\tau-2],\quad
\text{calibrate threshold on } \tau-1,\quad
\text{test on } \tau.
$$

This separates training from threshold selection. When the calibration step has too few positives, the code uses an $\epsilon$-fallback threshold rule.


![WF regimes](assets/08_wf_regimes.png)


### Walk-forward result table

| model | WF pooled F1 | WF macro F1 | pre-shock F1 | shock F1 | recovery F1 |
| --- | --- | --- | --- | --- | --- |
| SGC baseline | 0.338 | 0.309 | 0.535 | 0.016 | 0.095 |
| SGC + MLP | 0.530 | 0.408 | 0.731 | 0.013 | 0.105 |
| Best SGC WF | 0.713 | 0.481 | 0.822 | 0.000 | 0.259 |
| Best graph recovery | 0.679 | 0.471 | 0.767 | 0.056 | 0.360 |
| XGBoost WF | 0.834 | 0.634 | 0.902 | 0.000 | 0.472 |
| XGBoost + decay | 0.836 | 0.674 | 0.884 | 0.154 | 0.604 |


## The graph recovery trap

Graph models can perform well before the shock because they learn the local motifs of the pre-shock illicit economy.

After $\tau=43$, illicit actors re-enter through different local structures. The graph features learned before the shock become stale. This is **topological overfitting**:

$$
\text{good pre-shock motif memory} \;\not\Rightarrow\; \text{good post-shock generalization}.
$$

The evidence is the recovery gap: XGBoost recovers substantially better than SGC variants, even though graph models can be competitive pre-shock.


## Temporal decay

The temporal-decay ablation in `source/evaluation/ablation_validation.py` weights old training examples less:

$$
w_i \propto \exp\{-\lambda(\tau-t_i)\}\,c(y_i),
$$

where $c(y_i)$ is the class-imbalance multiplier. The mathematical idea is simple: if the graph regime has changed, old topology should not dominate the loss.


![Temporal decay](assets/09_temporal_decay.png)


## The final MLP-head experiment

The late experiment tried to improve the graph head on top of multiscale SGC.

The proposed residual idea was tested, but the result was clear:

- wide residual heads overfit and underperform;
- small residual heads still underperform;
- the useful change is **LayerNorm + SiLU + a small non-residual MLP**.

The winning graph-head recipe is therefore:

```python
use_mlp_head = True
use_layernorm = True
activation = "silu"
use_residual = False
mlp_hidden = (64, 64)
mlp_dropout = 0.4
sgc_k = 3
use_directional_prop = True
topo_injection_mode = "late"
use_pca = True
pca_variance = 0.98
sgc_lr = 0.01
sgc_weight_decay = 0.0005
```


![MLP validation deltas](assets/10_deep_mlp_validation_deltas.png)


### Validation deltas versus the old Grid MLP benchmark

| Metric | Old | run0 | Δ0 | run1 | Δ1 | run2 | Δ2 | run3 | Δ3 | sweepA | ΔA | sweepB | ΔB | sweepC | ΔC | sweepD | ΔD |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Val Macro F1 | 0.828500 | 0.813609 | -0.014891 | 0.840563 | +0.012063 | 0.822489 | -0.006011 | 0.812086 | -0.016414 | 0.821978 | -0.006522 | 0.811702 | -0.016798 | 0.811702 | -0.016798 | 0.821589 | -0.006911 |
| Val Pooled F1 | 0.877750 | 0.865486 | -0.012264 | 0.883412 | +0.005662 | 0.865184 | -0.012566 | 0.857414 | -0.020336 | 0.869615 | -0.008135 | 0.872116 | -0.005634 | 0.864910 | -0.012840 | 0.868935 | -0.008815 |
| Val Macro PR-AUC | 0.864750 | 0.848955 | -0.015795 | 0.836870 | -0.027880 | 0.875940 | +0.011190 | 0.869662 | +0.004912 | 0.880434 | +0.015684 | 0.880434 | +0.015684 | 0.880913 | +0.016163 | 0.880913 | +0.016163 |
| Val Pooled PR-AUC | 0.934667 | 0.919305 | -0.015362 | 0.869245 | -0.065422 | 0.946772 | +0.012105 | 0.938895 | +0.004228 | 0.943038 | +0.008371 | 0.938798 | +0.004132 | 0.935924 | +0.001257 | 0.938449 | +0.003783 |


![Phase sweeps](assets/11_phase_sweeps.png)


## What the final MLP head did and did not prove

**It did prove:**

- LayerNorm + SiLU improves graph-head ranking metrics over the old graph MLP baseline.
- Small heads generalize better than wide heads.
- Residual connections are not automatically beneficial in this setting.

**It did not prove:**

- that the graph model beats the best tabular baseline;
- that F1 improves under the fixed $0.5$ threshold;
- that architecture alone solves the $\tau=43$ label-deprivation problem.

The final claim should be precise: **we improved the graph MLP ranking performance, but the global benchmark is still dominated by tree-based tabular models.**


## Final conclusions

1. The Elliptic Bitcoin task is dominated by temporal non-stationarity and class imbalance.
2. $\tau=43$ is mainly a prior-shift event: the illicit class nearly disappears.
3. $\tau\ge 44$ creates the harder recovery problem: illicit actors return with different micro-structure.
4. SGC and multiscale graph features help, but deep graph propagation risks topological overfitting.
5. PCA regularizes deep propagation and partially rescues $K=3$.
6. Temporal decay is the most principled fix for stale topology.
7. The best final graph head is small, normalized, and non-residual.
8. XGBoost remains the strongest overall model, which is an important negative result for the graph-learning hypothesis.


## Implementation map

Important source files checked for this presentation:

| File | Role |
|---|---|
| `source/data/load_dataset.py` | loads Elliptic data and validates temporal edge integrity |
| `source/data/build_graph.py` | builds per-timestep graphs, scales features, injects topology, applies PCA |
| `source/models/layers.py` | SGC propagation, multiscale concatenation, directional channels |
| `source/models/classifier.py` | MLP head, LayerNorm, SiLU/ReLU validation, residual projection |
| `source/evaluation/validation.py` | static and walk-forward validation, threshold calibration |
| `source/evaluation/ablation_validation.py` | temporal decay and additional walk-forward ablations |
| `source/sweep.py` | experiment orchestration and result schema |
| `source/reporting/results/*.md` | written analyses used to shape the presentation narrative |


In [ ]:
# Reproducibility entry point
# The notebook's plots were generated by:
#   python source/reporting/build_presentation_notebook.py
#
# Main result folders:
#   results/final_aggregated_results.csv
#   results/final_aggregated_timesteps.csv
#   results/deep_res_mlp_results/sweep_phaseA
#   results/deep_res_mlp_results/sweep_phaseB
#   results/deep_res_mlp_results/sweep_phaseC
#   results/deep_res_mlp_results/sweep_phaseD
